[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/07_learning_to_rank_multitask/07_proyecto_ranker_multiobjetivo.ipynb)

# Módulo 07 · Proyecto: ranker multi-objetivo para la home de CineMatch
### Proyecto — Curso «Sistemas de Recomendación: de cero a nivel Netflix»

| | |
|---|---|
| **Nivel** | 🟠 Avanzado → 🔴 Experto |
| **Duración estimada** | 5–6 h |
| **GPU recomendada** | L4 / A100 (T4 vale con FAST_DEV_RUN) |
| **Unidades de Colab estimadas** | ~6–12 |
| **Prerrequisitos** | Lección 07 (y 06) |

> En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Todo el notebook
> funciona también en CPU con `FAST_DEV_RUN = True` (más lento, mismos conceptos).

## 🏢 Contexto de negocio
CineMatch lanza **CineMatch Clips**: una fila de trailers y clips cortos en la home. Producto tiene tres objetivos:
- 🖱️ **clic** (abrir el clip) — engagement inmediato,
- ⏱️ **long view** (verlo entero o casi) — señal de interés real, la que mejor predice que luego se vea la película,
- ❤️ **like** — satisfacción explícita, escasa pero valiosa.

Hasta ahora había un modelo de CTR (módulo 06). Tu misión: un **ranker multi-tarea** y una **fórmula de fusión** que maximice una métrica de negocio sin hundir ningún objetivo, más un **baseline LambdaMART**. Como CineMatch aún no tiene logs de clips, usamos los logs públicos de **KuaiRand-Pure** (Kuaishou), que tienen exactamente estas señales.

## 📦 Dataset
KuaiRand-Pure (Gao et al., CIKM 2022; Zenodo `records/10439422`). Logs estándar del 8-abr al 8-may-2022 con `is_click`, `long_view`, `is_like`, features de usuario y de vídeo. Split temporal 80/10/10 por `time_ms`. Fallback sintético si no hay red.

## ✅ Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | MMoE y PLE implementados (forward con puertas) | Tests de forma incluidos pasan |
| 2 | Tabla AUC por tarea: Single-task, Shared-Bottom, MMoE, PLE | MMoE o PLE ≥ Shared-Bottom en ≥ 2 de 3 tareas |
| 3 | Frente de Pareto clic–like (≥ 3 pesos) | Gráfico + comentario sobre el *seesaw* |
| 4 | LambdaMART (LightGBM) con etiqueta graduada de negocio | NDCG@10 de negocio reportado |
| 5 | Fusión de scores tuneada en **validación** y evaluada en **test** | Comparada con ordenar por cada tarea y con LambdaMART; objetivo orientativo: ≥ mejor tarea individual (si no se cumple, explica por qué) |
| 6 | Nota de decisión (≤ 10 líneas) para Producto | Pesos elegidos, trade-offs, plan de A/B |

In [ ]:
!pip install -q lightgbm

In [ ]:
import random, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
import os, urllib.request
FAST_DEV_RUN = True
MAX_ROWS = 600_000 if FAST_DEV_RUN else None
EPOCHS_MTL = 1 if FAST_DEV_RUN else 3

In [ ]:
# === KuaiRand-Pure (Kuaishou, CIKM 2022) con fallback sintético ==============
import tarfile, glob
KUAI_URL = "https://zenodo.org/records/10439422/files/KuaiRand-Pure.tar.gz"   # ~200 MB (md5 0820331067a3784d9691136f772b35a7)
TASKS = ["is_click", "long_view", "is_like"]

def synthetic_kuairand(n_users=1500, n_videos=2000, n_rows=400_000, seed=42):
    """Mismo esquema que KuaiRand-Pure. Tres tareas correlacionadas pero NO idénticas,
    que dependen de IDs (afinidad latente) y de features observables (actividad, tipo, música, duración, tab)."""
    rng = np.random.default_rng(seed)
    d = 8
    U, V = rng.normal(size=(n_users, d)), rng.normal(size=(n_videos, d))
    a_lv, a_like = rng.normal(size=d), rng.normal(size=d)
    act = rng.integers(0, 4, n_users); music = rng.integers(0, 10, n_videos); upl = rng.integers(0, 4, n_videos)
    author = rng.integers(0, 400, n_videos); author_q = rng.normal(0, 0.6, 400)
    vid_pop = rng.zipf(1.3, n_videos).clip(max=200).astype(float); vid_pop /= vid_pop.sum()
    u = rng.integers(0, n_users, n_rows); v = rng.choice(n_videos, n_rows, p=vid_pop)
    dur = rng.lognormal(10.5, 0.8, n_videos)                  # ms
    tab = rng.integers(0, 4, n_rows)
    aff = (U[u] * V[v]).sum(1) / np.sqrt(d)
    z_click = -0.9 + 0.9 * aff + np.array([0.6, 0.3, 0.0, -0.5])[act[u]] + author_q[author[v]] - 0.6 * (tab == 2) + 0.4 * (tab == 0)
    click = rng.random(n_rows) < 1 / (1 + np.exp(-z_click))
    z_lv = -0.3 + 0.7 * aff + 0.8 * (V[v] @ a_lv) / 3 - 0.9 * (np.log(dur[v]) - 10.5) + np.array([0.5, 0.0, -0.3, 0.2])[upl[v]]
    long_view = click & (rng.random(n_rows) < 1 / (1 + np.exp(-z_lv)))
    z_like = -3.0 + 0.5 * aff + 1.0 * ((U[u] * V[v]) @ a_like) / 3 + 0.25 * (music[v] - 4.5) / 2.9 + np.array([0.8, 0.4, 0.0, -0.4])[act[u]]
    like = click & (rng.random(n_rows) < 1 / (1 + np.exp(-z_like)))
    t = np.sort(rng.integers(1_649_376_000_000, 1_649_376_000_000 + 30 * 86_400_000, n_rows))
    logs = pd.DataFrame({"user_id": u, "video_id": v, "time_ms": t, "tab": tab,
                         "hourmin": (t // 3_600_000 % 24) * 100,
                         "is_click": click.astype(int), "long_view": long_view.astype(int), "is_like": like.astype(int)})
    users = pd.DataFrame({"user_id": np.arange(n_users),
                          "user_active_degree": np.array(["full_active", "high_active", "middle_active", "low_active"])[act],
                          "follow_user_num_range": rng.choice(["0", "(0,10]", "(10,50]", "(50,100]"], n_users),
                          "register_days_range": rng.choice(["15-30", "31-60", "61-90", "91-180", "181-365", "366-730", "730+"], n_users)})
    videos = pd.DataFrame({"video_id": np.arange(n_videos), "author_id": author,
                           "video_type": rng.choice(["NORMAL", "AD"], n_videos, p=[0.97, 0.03]),
                           "upload_type": np.array(["ShortImport", "Web", "Kmovie", "FollowShoot"])[upl],
                           "music_type": music, "video_duration": dur})
    return logs, users, videos

def load_kuairand(data_dir="data", max_rows=None):
    os.makedirs(data_dir, exist_ok=True)
    try:
        tgz = os.path.join(data_dir, "KuaiRand-Pure.tar.gz")
        if not glob.glob(os.path.join(data_dir, "**", "log_standard_4_08_to_4_21_pure.csv"), recursive=True):
            if not os.path.exists(tgz):
                print("Descargando KuaiRand-Pure de Zenodo (~200 MB)…")
                urllib.request.urlretrieve(KUAI_URL, tgz)
            with tarfile.open(tgz) as tf:
                tf.extractall(data_dir)
        f = lambda name: glob.glob(os.path.join(data_dir, "**", name), recursive=True)[0]
        logs = pd.concat([pd.read_csv(f("log_standard_4_08_to_4_21_pure.csv")),
                          pd.read_csv(f("log_standard_4_22_to_5_08_pure.csv"))], ignore_index=True)
        users = pd.read_csv(f("user_features_pure.csv"))
        videos = pd.read_csv(f("video_features_basic_pure.csv"))
        print(f"✅ KuaiRand-Pure real: {len(logs):,} impresiones")
    except Exception as e:
        print(f"⚠️ No se pudo cargar KuaiRand ({type(e).__name__}). Uso datos SINTÉTICOS con el mismo esquema.")
        logs, users, videos = synthetic_kuairand()
    logs = logs.sort_values("time_ms").reset_index(drop=True)
    if max_rows and len(logs) > max_rows:
        logs = logs.iloc[-max_rows:].reset_index(drop=True)      # nos quedamos con lo más reciente
    return logs, users, videos

KUAI_CATS = ["user_id", "video_id", "author_id", "tab", "hour", "user_active_degree",
             "follow_user_num_range", "register_days_range", "video_type", "upload_type", "music_type", "dur_bucket"]

def prepare_kuairand(logs, users, videos):
    df = logs.merge(users[["user_id", "user_active_degree", "follow_user_num_range", "register_days_range"]], on="user_id", how="left")
    df = df.merge(videos[["video_id", "author_id", "video_type", "upload_type", "music_type", "video_duration"]], on="video_id", how="left")
    df["hour"] = (df.hourmin // 100).astype(int)
    df["dur_bucket"] = pd.qcut(df.video_duration.fillna(df.video_duration.median()).rank(method="first"), 10, labels=False)
    n = len(df); i1, i2 = int(0.8 * n), int(0.9 * n)
    split = np.array(["train"] * i1 + ["val"] * (i2 - i1) + ["test"] * (n - i2))   # temporal (ordenado por time_ms)
    X, field_dims = [], []
    for c in KUAI_CATS:
        col = df[c].astype(str)
        vocab = pd.Index(col[split == "train"].unique())
        X.append(vocab.get_indexer(col) + 1); field_dims.append(len(vocab) + 1)     # OOV → 0
    X = (np.stack(X, 1) + np.cumsum([0] + field_dims[:-1])).astype(np.int32)
    Y = df[TASKS].values.astype(np.float32)
    return df, X, Y, field_dims, split

In [ ]:
logs, kusers, kvideos = load_kuairand(max_rows=MAX_ROWS)
kdf, KX, KY, kfield_dims, ksplit = prepare_kuairand(logs, kusers, kvideos)
Ktr, Kva, Kte = (ksplit == s for s in ["train", "val", "test"])
GAIN_W = np.array([1.0, 2.0, 4.0])          # ganancia de negocio: clic + 2·long_view + 4·like
print(KX.shape, dict(zip(TASKS, KY.mean(0).round(4))))

## Paso 1 — Implementa MMoE y PLE
Te damos `mlp`, `CatEncoder`, `SharedBottom` y `MTLNet`. Completa `MMoE.forward` y `CGCLayer.forward`. Ambos deben devolver `(logits [B, T], gates)`.

In [ ]:
def mlp(dims, dropout=0.0, last_act=True):
    layers = []
    for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
        layers.append(nn.Linear(a, b))
        if last_act or i < len(dims) - 2:
            layers += [nn.ReLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers)

class CatEncoder(nn.Module):
    def __init__(self, field_dims, k=16):
        super().__init__()
        self.emb = nn.Embedding(sum(field_dims), k); nn.init.normal_(self.emb.weight, std=0.01)
        self.d_out = len(field_dims) * k
    def forward(self, x):
        return self.emb(x).flatten(1)

class SharedBottom(nn.Module):
    def __init__(self, d_in, n_tasks, bottom=(256, 128), tower=(64,)):
        super().__init__()
        self.bottom = mlp([d_in, *bottom])
        self.towers = nn.ModuleList([mlp([bottom[-1], *tower, 1], last_act=False) for _ in range(n_tasks)])
    def forward(self, h):
        z = self.bottom(h)
        return torch.cat([t(z) for t in self.towers], 1), None

class MTLNet(nn.Module):
    def __init__(self, encoder, core):
        super().__init__(); self.encoder, self.core = encoder, core
    def forward(self, x):
        return self.core(self.encoder(x))

class MMoE(nn.Module):
    def __init__(self, d_in, n_tasks, n_experts=6, expert=(256, 128), tower=(64,)):
        super().__init__()
        self.experts = nn.ModuleList([mlp([d_in, *expert]) for _ in range(n_experts)])
        self.gates = nn.ModuleList([nn.Linear(d_in, n_experts) for _ in range(n_tasks)])
        self.towers = nn.ModuleList([mlp([expert[-1], *tower, 1], last_act=False) for _ in range(n_tasks)])
    def forward(self, h):
        # TODO: apila expertos [B, E, d]; para cada tarea, softmax de su puerta, mezcla y torre
        raise NotImplementedError

class CGCLayer(nn.Module):
    def __init__(self, d_in, d_out, n_tasks, n_spec=2, n_shared=2, last=False):
        super().__init__()
        self.T = n_tasks
        self.spec = nn.ModuleList([nn.ModuleList([mlp([d_in, d_out]) for _ in range(n_spec)]) for _ in range(n_tasks)])
        self.shared = nn.ModuleList([mlp([d_in, d_out]) for _ in range(n_shared)])
        self.gates = nn.ModuleList([nn.Linear(d_in, n_spec + n_shared) for _ in range(n_tasks)])
        self.shared_gate = None if last else nn.Linear(d_in, n_tasks * n_spec + n_shared)
    def forward(self, xs, x_sh):
        # TODO: devuelve (salidas por tarea, nueva entrada compartida o None, puertas)
        raise NotImplementedError

class PLE(nn.Module):
    def __init__(self, d_in, n_tasks, levels=2, d_hidden=128, n_spec=2, n_shared=2, tower=(64,)):
        super().__init__()
        self.layers = nn.ModuleList([CGCLayer(d_in if l == 0 else d_hidden, d_hidden, n_tasks, n_spec, n_shared,
                                              last=(l == levels - 1)) for l in range(levels)])
        self.towers = nn.ModuleList([mlp([d_hidden, *tower, 1], last_act=False) for _ in range(n_tasks)])
        self.T = n_tasks
    def forward(self, h):
        xs, x_sh = [h] * self.T, h
        for layer in self.layers:
            xs, x_sh, gates = layer(xs, x_sh)
        return torch.cat([tw(x) for tw, x in zip(self.towers, xs)], 1), torch.stack(gates, 0)

def build_mtl(kind, field_dims, n_tasks, k=16):
    enc = CatEncoder(field_dims, k)
    core = {"Shared-Bottom": lambda: SharedBottom(enc.d_out, n_tasks), "MMoE": lambda: MMoE(enc.d_out, n_tasks),
            "PLE": lambda: PLE(enc.d_out, n_tasks)}[kind]()
    return MTLNet(enc, core)

def _test_shapes():
    x = torch.randn(4, 32)
    for core in [MMoE(32, 3), PLE(32, 3)]:
        out, g = core(x)
        assert out.shape == (4, 3) and g.shape[0] == 3 and torch.allclose(g.sum(-1), torch.ones_like(g.sum(-1)), atol=1e-5)
    print("✅ formas y puertas OK")
try:
    _test_shapes()
except NotImplementedError:
    print("⏳ Implementa MMoE.forward y CGCLayer.forward")

## Paso 2 — Entrenamiento y evaluación por tarea
Usa `train_mtl` (dado) y calcula AUC por tarea en test para Single-task, Shared-Bottom, MMoE y PLE.

In [ ]:
# === Entrenamiento multi-tarea con pesos fijos o "uncertainty weighting" =====
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def predict_mtl(model, X, bs=65536):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        logits, _ = model(torch.as_tensor(X[i:i + bs], device=device).long())
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out)

def train_mtl(model, Xtr, Ytr, Xva, Yva, task_w=None, epochs=2, bs=4096, lr=1e-3, weighting="fixed", verbose=False):
    T = Ytr.shape[1]
    task_w = torch.tensor(task_w if task_w is not None else [1.0] * T, device=device)
    model = model.to(device)
    log_vars = torch.zeros(T, device=device, requires_grad=True)          # Kendall et al. 2018
    params = list(model.parameters()) + ([log_vars] if weighting == "uncertainty" else [])
    opt = torch.optim.Adam(params, lr=lr)
    Xt = torch.as_tensor(Xtr, device=device); Yt = torch.as_tensor(Ytr, device=device)
    best, best_state = -np.inf, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt), device=device)
        for s in range(0, len(Xt), bs):
            i = perm[s:s + bs]
            logits, _ = model(Xt[i].long())
            l_t = F.binary_cross_entropy_with_logits(logits, Yt[i], reduction="none").mean(0)   # [T]
            loss = (task_w * l_t).sum() if weighting == "fixed" else (torch.exp(-log_vars) * l_t + log_vars).sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        P = predict_mtl(model, Xva)
        aucs = [roc_auc_score(Yva[:, t], P[:, t]) for t in range(T)]
        if verbose: print(f"  época {ep+1}: AUC val", np.round(aucs, 4))
        if np.mean(aucs) > best:
            best = np.mean(aucs); best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

In [ ]:
# TODO: entrena los 4 enfoques y construye la tabla de AUC por tarea

## Paso 3 — Frente de Pareto (clic vs like)
Barre `task_w=[1, 1, w]` con ≥ 3 valores de `w` para al menos dos arquitecturas. Grafica AUC clic vs AUC like.

In [ ]:
# TODO

## Paso 4 — Baseline LambdaMART
Cada **usuario en un día** es una consulta; la etiqueta graduada es la ganancia de negocio entera `clic + 2·lv + 4·like` (0–7). Features para LightGBM: las categóricas de `KUAI_CATS` (como `category`) + **features de conteo point-in-time** que tú construyas (p. ej. CTR previo del vídeo y del autor con suavizado; ver proyecto 06).

In [ ]:
import lightgbm as lgb
def business_ndcg(df_scores: pd.DataFrame, score_col: str, k=10) -> float:
    """df_scores con columnas qid, gain y score_col. NDCG@k de negocio promedio por consulta."""
    # TODO
    raise NotImplementedError

## Paso 5 — Fusión tuneada en validación
Busca en una rejilla los pesos de $s=\log\hat p_\text{clic} + w_\text{lv}\log\hat p_\text{lv} + w_\text{like}\log\hat p_\text{like}$ **en validación**, y reporta el resultado en **test** (¡no tunees en test!). Compara con ordenar solo por cada tarea y con LambdaMART.

In [ ]:
# TODO

---
# ⛔ SPOILER — Solución de referencia
Intenta resolverlo primero.

In [ ]:
class MMoE(nn.Module):
    def __init__(self, d_in, n_tasks, n_experts=6, expert=(256, 128), tower=(64,)):
        super().__init__()
        self.experts = nn.ModuleList([mlp([d_in, *expert]) for _ in range(n_experts)])
        self.gates = nn.ModuleList([nn.Linear(d_in, n_experts) for _ in range(n_tasks)])
        self.towers = nn.ModuleList([mlp([expert[-1], *tower, 1], last_act=False) for _ in range(n_tasks)])
    def forward(self, h):
        E = torch.stack([e(h) for e in self.experts], 1)
        outs, gates = [], []
        for gate, tower in zip(self.gates, self.towers):
            g = torch.softmax(gate(h), -1)
            outs.append(tower((g.unsqueeze(-1) * E).sum(1))); gates.append(g)
        return torch.cat(outs, 1), torch.stack(gates, 0)

class CGCLayer(nn.Module):
    def __init__(self, d_in, d_out, n_tasks, n_spec=2, n_shared=2, last=False):
        super().__init__()
        self.T = n_tasks
        self.spec = nn.ModuleList([nn.ModuleList([mlp([d_in, d_out]) for _ in range(n_spec)]) for _ in range(n_tasks)])
        self.shared = nn.ModuleList([mlp([d_in, d_out]) for _ in range(n_shared)])
        self.gates = nn.ModuleList([nn.Linear(d_in, n_spec + n_shared) for _ in range(n_tasks)])
        self.shared_gate = None if last else nn.Linear(d_in, n_tasks * n_spec + n_shared)
    def forward(self, xs, x_sh):
        sh = [e(x_sh) for e in self.shared]
        outs, gates, all_spec = [], [], []
        for t in range(self.T):
            sp = [e(xs[t]) for e in self.spec[t]]; all_spec += sp
            g = torch.softmax(self.gates[t](xs[t]), -1)
            outs.append((g.unsqueeze(-1) * torch.stack(sp + sh, 1)).sum(1)); gates.append(g)
        new_sh = None
        if self.shared_gate is not None:
            g = torch.softmax(self.shared_gate(x_sh), -1)
            new_sh = (g.unsqueeze(-1) * torch.stack(all_spec + sh, 1)).sum(1)
        return outs, new_sh, gates

_test_shapes()

In [ ]:
res, models, preds = {}, {}, {}
for kind in ["Shared-Bottom", "MMoE", "PLE"]:
    torch.manual_seed(seed)
    models[kind] = train_mtl(build_mtl(kind, kfield_dims, 3), KX[Ktr], KY[Ktr], KX[Kva], KY[Kva], epochs=EPOCHS_MTL)
    preds[kind] = {s: predict_mtl(models[kind], KX[m]) for s, m in [("val", Kva), ("test", Kte)]}
    res[kind] = {t: roc_auc_score(KY[Kte][:, i], preds[kind]["test"][:, i]) for i, t in enumerate(TASKS)}
single_val, single_test = np.zeros((Kva.sum(), 3)), np.zeros((Kte.sum(), 3))
for i, t in enumerate(TASKS):
    torch.manual_seed(seed)
    m = train_mtl(build_mtl("Shared-Bottom", kfield_dims, 1), KX[Ktr], KY[Ktr][:, [i]], KX[Kva], KY[Kva][:, [i]], epochs=EPOCHS_MTL)
    single_val[:, i] = predict_mtl(m, KX[Kva])[:, 0]; single_test[:, i] = predict_mtl(m, KX[Kte])[:, 0]
res["Single-task"] = {t: roc_auc_score(KY[Kte][:, i], single_test[:, i]) for i, t in enumerate(TASKS)}
preds["Single-task"] = {"val": single_val, "test": single_test}
auc_table = pd.DataFrame(res).T
display(auc_table.round(4))
wins = {k: int((auc_table.loc[k] >= auc_table.loc["Shared-Bottom"]).sum()) for k in ["MMoE", "PLE"]}
print("Tareas en las que igualan o superan a Shared-Bottom:", wins)

In [ ]:
pareto = []
for kind in ["Shared-Bottom", "PLE"]:
    for w in ([0.3, 1.0, 5.0] if FAST_DEV_RUN else [0.1, 0.3, 1.0, 3.0, 10.0]):
        torch.manual_seed(seed)
        m = train_mtl(build_mtl(kind, kfield_dims, 3), KX[Ktr], KY[Ktr], KX[Kva], KY[Kva], task_w=[1, 1, w], epochs=EPOCHS_MTL)
        P = predict_mtl(m, KX[Kte])
        pareto.append((kind, w, roc_auc_score(KY[Kte][:, 0], P[:, 0]), roc_auc_score(KY[Kte][:, 2], P[:, 2])))
pf = pd.DataFrame(pareto, columns=["modelo", "w_like", "auc_click", "auc_like"])
plt.figure(figsize=(6, 4.2))
for kind, g in pf.groupby("modelo"):
    plt.plot(g.auc_click, g.auc_like, "o-", label=kind)
    for _, r in g.iterrows(): plt.annotate(f"w={r.w_like:g}", (r.auc_click, r.auc_like), fontsize=7)
plt.xlabel("AUC clic"); plt.ylabel("AUC like"); plt.legend(); plt.title("Frente de Pareto"); plt.show()

In [ ]:
# LambdaMART: consulta = usuario × día; features categóricas + conteos point-in-time
kdf["day"] = (kdf.time_ms // 86_400_000).astype(int)
kdf["gain"] = KY @ GAIN_W
kdf["qid"] = kdf.groupby(["user_id", "day"]).ngroup()
for key in ["video_id", "author_id"]:                           # CTR previo suavizado (solo pasado)
    g = kdf.groupby(key)
    cnt = g.cumcount(); clk = g.is_click.cumsum() - kdf.is_click
    prior = kdf.is_click[Ktr].mean()
    kdf[f"{key}_cnt_pit"] = np.log1p(cnt); kdf[f"{key}_ctr_pit"] = (clk + 20 * prior) / (cnt + 20)
LGB_FEATS = KUAI_CATS + [c for c in kdf.columns if c.endswith("_pit")]
Xl = kdf[LGB_FEATS].copy()
for c in KUAI_CATS:
    Xl[c] = pd.Categorical(pd.Index(Xl.loc[Ktr, c].astype(str).unique()).get_indexer(Xl[c].astype(str)) + 1)

def grouped(mask):
    idx = np.where(mask)[0]; idx = idx[np.argsort(kdf.qid.values[idx], kind="mergesort")]
    return idx, kdf.qid.values[idx]

itr, qtr = grouped(Ktr); iva, qva = grouped(Kva)
ranker = lgb.LGBMRanker(objective="lambdarank", n_estimators=600, learning_rate=0.05, num_leaves=63,
                        min_child_samples=100, label_gain=[2 ** i - 1 for i in range(8)],
                        random_state=seed, verbose=-1)
ranker.fit(Xl.iloc[itr], kdf.gain.values[itr].astype(int), group=np.unique(qtr, return_counts=True)[1],
           eval_set=[(Xl.iloc[iva], kdf.gain.values[iva].astype(int))], eval_group=[np.unique(qva, return_counts=True)[1]],
           eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)])

def business_ndcg(df_scores: pd.DataFrame, score_col: str, k=10) -> float:
    vals = []
    for _, g in df_scores.groupby("qid"):
        if len(g) < 3 or g.gain.sum() == 0: continue
        top = g.nlargest(k, score_col).gain.values
        ideal = np.sort(g.gain.values)[::-1][:k]
        disc = 1 / np.log2(np.arange(2, k + 2))
        vals.append(((2 ** top - 1) * disc[:len(top)]).sum() / ((2 ** ideal - 1) * disc[:len(ideal)]).sum())
    return float(np.mean(vals))

ev = {s: kdf.loc[m, ["qid", "gain"]].copy() for s, m in [("val", Kva), ("test", Kte)]}
ev["test"]["lambdamart"] = ranker.predict(Xl[Kte]); ev["val"]["lambdamart"] = ranker.predict(Xl[Kva])
print("LambdaMART NDCG@10 de negocio (test):", round(business_ndcg(ev["test"], "lambdamart"), 4))

In [ ]:
# Fusión: rejilla en VALIDACIÓN, evaluación final en TEST
grid = [0.0, 0.5, 1.0, 2.0, 4.0]
P_val, P_test = preds["PLE"]["val"], preds["PLE"]["test"]
fuse = lambda P, a, b: np.log(P[:, 0] + 1e-9) + a * np.log(P[:, 1] + 1e-9) + b * np.log(P[:, 2] + 1e-9)
best = max(((a, b) for a in grid for b in grid),
           key=lambda ab: business_ndcg(ev["val"].assign(s=fuse(P_val, *ab)), "s"))
print("Pesos elegidos en validación (w_lv, w_like):", best)
final = {}
for i, t in enumerate(TASKS):
    final[f"solo {t} (PLE)"] = business_ndcg(ev["test"].assign(s=P_test[:, i]), "s")
final["LambdaMART"] = business_ndcg(ev["test"], "lambdamart")
final[f"Fusión PLE w={best}"] = business_ndcg(ev["test"].assign(s=fuse(P_test, *best)), "s")
final[f"Fusión Single-task w={best}"] = business_ndcg(ev["test"].assign(s=fuse(preds['Single-task']['test'], *best)), "s")
s = pd.Series(final).sort_values()
s.plot.barh(figsize=(7, 3.5), color="#4c72b0", title="NDCG@10 de negocio en test"); plt.xlim(s.min() * 0.97, s.max() * 1.01); plt.show()
print(s.round(4))

### Nota de decisión (ejemplo)
> Recomendamos servir **PLE** (una sola red, 3 cabezas) con fusión $\log p_\text{clic} + w_\text{lv}\log p_\text{lv} + w_\text{like}\log p_\text{like}$ con los pesos elegidos en validación. Frente a ordenar por clic, mejora la NDCG de negocio offline sin empeorar de forma relevante el AUC de clic (ver frente de Pareto). LambdaMART es un baseline sólido y barato; lo mantenemos como *fallback*. Plan: A/B con 3 brazos (solo-clic, fusión elegida, fusión con w_like×2), métrica principal = minutos vistos de películas tras ver un clip, *guardrails*: CTR de la fila y tasa de `is_hate`. Monitorizar calibración por cabeza.

## 🚀 Retos extra
1. Añade `is_follow`, `is_comment`, `is_forward` y la regresión de `play_time_ms` (pérdida MSE sobre log1p) como tareas. ¿Aparece *seesaw*?
2. Implementa **uncertainty weighting** y **PCGrad** y compáralos en el frente de Pareto.
3. Usa el log **aleatorio** de KuaiRand (`log_random_4_22_to_5_08_pure.csv`) como test **insesgado**: ¿cambia el ranking de modelos?
4. Añade un *shallow tower* de posición/`tab` estilo YouTube 2019 que se apague en serving.
5. Destila LambdaMART en la red (usa sus scores como *soft labels* de una tarea extra).

## 🤔 Reflexión (producción / MLOps)
- ¿Cómo versionarías la **fórmula de fusión** (pesos) separada del modelo, para cambiarla sin reentrenar? (Config en un registry, módulo 17).
- ¿Qué monitorizarías por cabeza (calibración, distribución de scores) y qué alerta dispararía un rollback?
- Si una tarea (like) cambia de definición en la app, ¿cómo evitas que el reentreno rompa la fusión?